# Lab 15: Evolución Temporal con Trotterización

Simular la evolución temporal $|\psi(t)\rangle = e^{-iHt}|\psi_0\rangle$ de un sistema cuántico es una de las tareas más naturales para un ordenador cuántico. El problema es que en general $H = H_1 + H_2 + \ldots$ y los sumandos **no conmutan**, por lo que $e^{-iHt} \neq e^{-iH_1 t}e^{-iH_2 t}$.

La **fórmula de Trotter** aproxima la exponencial conjunta descomponiendo el tiempo total en $n$ pasos pequeños $\delta t = t/n$:

$$e^{-iHt} \approx \left(e^{-iH_1\delta t}e^{-iH_2\delta t}\right)^n + O\!\left(\frac{t^2}{n}\right)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator, state_fidelity, DensityMatrix
from qiskit.primitives import StatevectorEstimator

# Hamiltoniano no-conmutante: H = XX + 0.5*ZI + 0.5*IZ
H_total = SparsePauliOp.from_list([('XX', 1.0), ('ZI', 0.5), ('IZ', 0.5)])
H_mat   = H_total.to_matrix()
H1_mat  = SparsePauliOp('XX').to_matrix()
H2_mat  = SparsePauliOp.from_list([('ZI', 0.5), ('IZ', 0.5)]).to_matrix()

print("H = XX + 0.5·ZI + 0.5·IZ")
print(f"||[H1, H2]|| = {np.linalg.norm(H1_mat @ H2_mat - H2_mat @ H1_mat):.4f}  (≠ 0 → error Trotter)")

## 1. Evolución Exacta vs Trotter

Comparamos el operador unitario exacto $U(t) = e^{-iHt}$ con la aproximación Trotter de primer orden $U_n^{(1)}(t) = (e^{-iH_1 t/n} e^{-iH_2 t/n})^n$.

El error en norma de operador decrece como $\|U - U_n\| \sim t^2/n$.

In [ ]:
def exact_U(t): return expm(-1j * H_mat * t)
def trotter_U(t, n):
    dt = t / n
    step = expm(-1j * H1_mat * dt) @ expm(-1j * H2_mat * dt)
    return np.linalg.matrix_power(step, n)
def suzuki2_U(t, n):
    dt = t / n
    step = (expm(-1j * H1_mat * dt/2) @
            expm(-1j * H2_mat * dt) @
            expm(-1j * H1_mat * dt/2))
    return np.linalg.matrix_power(step, n)

t = 3.0
n_vals = [1, 2, 4, 8, 16, 32, 64]
U_ref = exact_U(t)
err_t1 = [np.linalg.norm(trotter_U(t, n) - U_ref) for n in n_vals]
err_s2 = [np.linalg.norm(suzuki2_U(t, n) - U_ref) for n in n_vals]

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(n_vals, err_t1, 'o-', color='steelblue', lw=2, label='Trotter 1° orden')
ax.loglog(n_vals, err_s2, 's-', color='darkorange', lw=2, label='Suzuki 2° orden')
ns = np.array(n_vals, dtype=float)
ax.loglog(ns, err_t1[0] * (ns[0]/ns)**1, 'b--', alpha=0.5, label='∝ 1/n')
ax.loglog(ns, err_s2[0] * (ns[0]/ns)**2, 'r--', alpha=0.5, label='∝ 1/n²')
ax.set_xlabel('Número de pasos n', fontsize=12)
ax.set_ylabel('Error ‖U_Trotter − U_exacta‖', fontsize=12)
ax.set_title(f'Convergencia Trotter, t={t}')
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

for n in [2, 8, 32]:
    print(f"n={n:2d} | Trotter: {trotter_U(t,n)-U_ref|np.linalg.norm:.4f} | Suzuki2: {suzuki2_U(t,n)-U_ref|np.linalg.norm:.4f}")


## 2. Evolución del Estado Cuántico

Aplicamos la evolución Trotter a un estado inicial concreto $|\psi_0\rangle = |01\rangle$ y comparamos la fidelidad del estado evolucionado con la referencia exacta.

In [ ]:
psi0 = np.zeros(4, dtype=complex)
psi0[1] = 1.0  # |01⟩

def evolve_state(psi, U): return U @ psi

t_vals = np.linspace(0, 5, 50)
fidelities_n1  = []
fidelities_n10 = []
fidelities_n50 = []

for t in t_vals:
    U_ref_t = exact_U(t)
    psi_ref = evolve_state(psi0, U_ref_t)
    for n, flist in [(1, fidelities_n1), (10, fidelities_n10), (50, fidelities_n50)]:
        psi_t = evolve_state(psi0, trotter_U(t, n))
        fid = abs(np.vdot(psi_ref, psi_t))**2
        flist.append(fid)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_vals, fidelities_n1,  lw=2, label='n=1',  color='crimson')
ax.plot(t_vals, fidelities_n10, lw=2, label='n=10', color='darkorange')
ax.plot(t_vals, fidelities_n50, lw=2, label='n=50', color='steelblue')
ax.axhline(0.99, color='gray', ls=':', lw=1.5, label='Fidelidad 0.99')
ax.set_xlabel('Tiempo t', fontsize=12)
ax.set_ylabel('Fidelidad F(|ψ_T⟩, |ψ_exacto⟩)', fontsize=12)
ax.set_title('Fidelidad de la Evolución Trotter vs Tiempo')
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

## 3. Circuito Cuántico de Trotterización

Implementamos el paso Trotter directamente como circuito Qiskit. Las compuertas $e^{-i\theta ZZ/2}$ se descomponen en CX + Rz + CX.

In [ ]:
def trotter_circuit_step(dt):
    """Un paso Trotter para H = XX + 0.5*ZI + 0.5*IZ."""
    qc = QuantumCircuit(2)
    # e^{-i XX dt}: H⊗H, CX, Rz(2dt), CX, H⊗H
    qc.h([0, 1])
    qc.cx(0, 1)
    qc.rz(2 * dt, 1)
    qc.cx(0, 1)
    qc.h([0, 1])
    # e^{-i 0.5 ZI dt}: Rz(dt, 0)
    qc.rz(dt, 0)
    # e^{-i 0.5 IZ dt}: Rz(dt, 1)
    qc.rz(dt, 1)
    return qc

t_test = 1.0
n_steps = 10
dt = t_test / n_steps

full_circ = QuantumCircuit(2)
full_circ.x(1)   # estado inicial |01⟩
for _ in range(n_steps):
    full_circ.compose(trotter_circuit_step(dt), inplace=True)

sv_trotter = Statevector(full_circ)
psi_exact  = exact_U(t_test) @ psi0
fid = abs(np.vdot(psi_exact, sv_trotter.data))**2
print(f"Circuito Trotter (n={n_steps}, t={t_test}):")
print(f"  Profundidad del circuito: {full_circ.depth()}")
print(f"  Fidelidad con evolución exacta: {fid:.6f}")

step_circ = trotter_circuit_step(dt)
step_circ.draw('mpl', style='clifford')

## 4. Orden de Error y Escalado

Verificamos empíricamente el orden del error Trotter estudiando cómo decrece $\|U_n - U\|$ al doblar $n$. Para el primer orden la razón de convergencia debe ser $\approx 2$; para Suzuki de segundo orden, $\approx 4$.

In [ ]:
t = 1.5
n_pairs = [(2, 4), (4, 8), (8, 16)]
U_ref_15 = exact_U(t)

print("Razón de convergencia (error con n) / (error con 2n):")
print(f"{'n':>4} | {'Trotter err':>12} | {'ratio':>6} | {'Suzuki2 err':>12} | {'ratio':>6}")
print("-" * 55)
prev_t, prev_s = None, None
for n in [2, 4, 8, 16]:
    et = np.linalg.norm(trotter_U(t, n) - U_ref_15)
    es = np.linalg.norm(suzuki2_U(t, n) - U_ref_15)
    rt = f"{prev_t/et:.2f}" if prev_t else "  —"
    rs = f"{prev_s/es:.2f}" if prev_s else "  —"
    print(f"{n:>4} | {et:>12.6f} | {rt:>6} | {es:>12.6f} | {rs:>6}")
    prev_t, prev_s = et, es

print("\nOrden esperado: Trotter ≈ 2×, Suzuki2 ≈ 4×")